In [ ]:
!pip install gensim nltk
import nltk
nltk.download('punkt', quiet=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 61.7 MB/s eta 0:00:00


True

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('cleaned_data.csv')
docs = df['clean_text'].astype(str).tolist()

print(f"Total Data: {len(docs)}")

Total Data: 20000


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(docs)

vocab = vectorizer.get_feature_names_out()
print("Vocabulary Size:", len(vocab))
print("Sample Vocab (10 pertama):", vocab[:10])
print("Matrix Shape:", X_bow.shape)

Vocabulary Size: 869
Sample Vocab (10 pertama): ['ability' 'able' 'accept' 'according' 'account' 'across' 'act' 'action'
 'activity' 'actually']
Matrix Shape: (20000, 869)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(norm=None, use_idf=True, smooth_idf=False)
X_tfidf = tfidf.fit_transform(docs)

vocab = tfidf.get_feature_names_out()
idf = tfidf.idf_

print("Vocabulary (10 pertama):", vocab[:10])

print("\nIDF Values (10 pertama):")
for word, val in zip(vocab[:10], idf[:10]):
    print(f"{word}: {round(val, 3)}")

print("\nTF-IDF Matrix Shape:", X_tfidf.shape)
tfidf_matrix = X_tfidf.toarray()
print("Sample Matrix (3 baris x 5 kolom):")
print(tfidf_matrix[:3, :5])

Vocabulary (10 pertama): ['ability' 'able' 'accept' 'according' 'account' 'across' 'act' 'action'
 'activity' 'actually']

IDF Values (10 pertama):
ability: 2.452
able: 2.48
accept: 2.482
according: 2.496
account: 2.468
across: 2.473
act: 2.464
action: 2.502
activity: 2.49
actually: 2.465

TF-IDF Matrix Shape: (20000, 869)
Sample Matrix (3 baris x 5 kolom):
[[0.         0.         0.         0.         0.        ]
 [2.45179334 0.         2.48192459 0.         0.        ]
 [2.45179334 0.         0.         0.         0.        ]]


In [ ]:
import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('punkt', quiet=True)  # fallback untuk kompatibilitas

True

In [ ]:
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize

tokenized_docs = [word_tokenize(doc) for doc in docs]

w2v_model = Word2Vec(
    sentences=tokenized_docs,
    vector_size=100,
    window=5,
    min_count=2,
    sg=1,
    epochs=10
)

def doc_vector(tokens, model):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(model.vector_size)

X_w2v = np.array([doc_vector(doc, w2v_model) for doc in tokenized_docs])

print("Word2Vec Shape:", X_w2v.shape)

Word2Vec Shape: (20000, 100)


In [ ]:
np.save('X_tfidf.npy', tfidf_matrix)
np.save('X_w2v.npy', X_w2v)

y = df['label'].map({'real': 0, 'fake': 1}).values
np.save('y_labels.npy', y)

print("File berhasil disimpan: X_tfidf.npy, X_w2v.npy, y_labels.npy")

File berhasil disimpan: X_tfidf.npy, X_w2v.npy, y_labels.npy
